🔷 Question 1: Build an End-to-End RAG + Agent System (25 Marks)
🧩 Scenario
You are an AI intern at an ed-tech company. Students frequently ask questions about:

Course policies (refunds, deadlines)
Lecture content (PDF notes)
Assignment deadlines
Their enrollment status
The company wants a single intelligent assistant that:

Answers questions using internal documents (PDFs, FAQs)
Fetches student-specific data (like enrollment or progress) using tools/APIs
Avoids hallucination and gives reliable answers
💻 Task
Design and implement a working prototype (pseudo-code or real code) for this system.

Your solution must include:

✅ 1. RAG Pipeline
How you will ingest and preprocess documents
Chunking strategy (with justification)
Embedding + vector store choice
Retrieval logic
How context is injected into the LLM
✅ 2. Agent Integration
Design an agent that decides:
When to use RAG
When to call a tool (e.g., get_student_status(student_id))
Show how tools are defined and used
✅ 3. End-to-End Flow
Write code or structured pseudo-code showing:

Input query
Retrieval step
Tool calling (if needed)
Final answer generation
✅ 4. Reliability Improvements
Show at least 2 techniques in code/design to:

Reduce hallucination
Improve answer quality
🎯 Expected Outcome
A working architecture/code that demonstrates:

RAG + Agent working together
Decision-making capability
Real-world practicality

In [3]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from transformers import pipeline

def load_pdf(path):
    try:
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            if page.extract_text():
                text += page.extract_text()
        return text
    except:
        print("⚠️ PDF not found, using dummy data")
        return """
        Refund policy: Students can request refund within 7 days.
        Assignments deadline must be followed strictly.
        Course access valid for 6 months.
        """

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i+chunk_size])
        i += chunk_size - overlap
    return chunks

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("docs")
except:
    pass
collection = client.create_collection("docs")

def store_chunks(chunks):
    for i, chunk in enumerate(chunks):
        emb = embedding_model.encode(chunk).tolist()
        collection.add(documents=[chunk], embeddings=[emb], ids=[str(i)])

def retrieve(query, k=3):
    q_emb = embedding_model.encode(query).tolist()
    results = collection.query(query_embeddings=[q_emb], n_results=k)
    return results["documents"][0]

def get_student_status(student_id):
    return f"Student {student_id} enrolled, progress 75%, next deadline in 5 days"

llm = pipeline("text-generation", model="google/flan-t5-base", max_new_tokens=200)

def agent_router(query):
    if "my" in query or "status" in query or "enrollment" in query:
        return "tool"
    return "rag"

def build_prompt(context, query):
    return f"""
Answer ONLY using the context below.
If not found, say "Not found".

Context:
{context}

Question:
{query}

Answer:
"""

def system(query, student_id=None):

    decision = agent_router(query)

    if decision == "tool":
        data = get_student_status(student_id)
        prompt = f"{data}\nQuestion: {query}\nAnswer:"
        return llm(prompt)[0]["generated_text"]

    else:
        docs = retrieve(query, k=5)
        context = " ".join(docs)

        prompt = build_prompt(context, query)
        answer = llm(prompt)[0]["generated_text"]

        if "Not found" in answer:
            answer = llm(prompt)[0]["generated_text"]

        return answer

text = load_pdf("course_policy.pdf")
chunks = chunk_text(text)
store_chunks(chunks)

print(system("What is refund policy?"))
print(system("What is my enrollment status?", student_id=101))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

⚠️ PDF not found, using dummy data


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer ONLY using the context below.
If not found, say "Not found".

Context:

        Refund policy: Students can request refund within 7 days.
        Assignments deadline must be followed strictly.
        Course access valid for 6 months.
        

Question:
What is refund policy?

Answer:

Student 101 enrolled, progress 75%, next deadline in 5 days
Question: What is my enrollment status?
Answer:


🔷 Question 2: Design a Multi-Agent Workflow with LangGraph (25 Marks)
🧩 Scenario
You are building an AI-powered customer support system for a fintech company.

The system must handle:

Transaction queries
Fraud detection flags
Refund requests
General FAQs
The company wants:

High accuracy
Step-by-step reasoning
Ability to retry if answer is incorrect
Modular, scalable architecture
💻 Task
Design and implement a multi-agent workflow using LangGraph (or similar framework).

✅ 1. Agent Design
Define at least 3 agents, such as:

Retrieval Agent
Reasoning/Answer Agent
Validation Agent
Explain briefly (in comments or code):

Each agent’s role
Input/output
✅ 2. Graph Workflow Implementation
Write code or pseudo-code to:

Define state
Add nodes (agents)
Define edges
Implement conditional logic
👉 Must include:

Retry loop if validation fails
Clear start and end states
✅ 3. State Management
Show how state evolves across steps:

Query
Context
Intermediate reasoning
Final answer
Validation flag
✅ 4. Task Delegation & Communication
Demonstrate:

How agents pass information
How decisions are made between agents
🎯 Expected Outcome
A clear multi-step, graph-based agent system that:

Handles complex queries
Demonstrates reasoning + validation
Uses proper orchestration

In [4]:
def init_state(query):
    return {
        "query": query,
        "context": [],
        "answer": "",
        "valid": False
    }

def retrieval_agent(state):
    print("[Retrieval Agent]")
    state["context"] = retrieve(state["query"], k=3)
    return state

def reasoning_agent(state):
    print("[Reasoning Agent]")
    context = " ".join(state["context"])

    prompt = f"""
Answer using ONLY the context.

Context:
{context}

Question:
{state['query']}

Answer:
"""
    state["answer"] = llm(prompt)[0]["generated_text"]
    return state

def validation_agent(state):
    print("[Validation Agent]")
    if "Not found" in state["answer"] or len(state["answer"]) < 20:
        state["valid"] = False
    else:
        state["valid"] = True
    return state

def workflow(query):

    state = init_state(query)

    state = retrieval_agent(state)

    state = reasoning_agent(state)

    state = validation_agent(state)

    if not state["valid"]:
        print("[Retrying Reasoning]")
        state = reasoning_agent(state)

    return state["answer"]

print(workflow("What is refund policy?"))

[Retrieval Agent]
[Reasoning Agent]


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Validation Agent]

Answer using ONLY the context.

Context:

        Refund policy: Students can request refund within 7 days.
        Assignments deadline must be followed strictly.
        Course access valid for 6 months.
        

Question:
What is refund policy?

Answer:
in. Why might have I checked and realized my students won/'n' forgotten those changes during their lunchroom sessions, while someone got on with your student by himself!, rather at breakfast. To tell her: the course might just make correction without knowing there had really an mistake that hadnif I found one but wasn? To help our team
